# <b>Perform Differential Expression (DE) Analysis</b>
The CEBPE-perturbed and control cell populations are compared via differential expression analysis to identify genes significantly upregulated or downregulated upon CEBPE overactivation.

---

## 1. Setup Environment

### 1.1. Import Libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import scanpy as sc

### 1.2. Define File Paths/Directories

In [ ]:
import config

# Clustered AnnData file
CLUSTERED_ANNDATA_FILE = config.PROCESSED_DATA_DIR / "03_Norman_2019_clustered.h5ad"
print("\nClustered AnnData file:\n", CLUSTERED_ANNDATA_FILE)

# Differential expression results save directory
# DE-Analyzed AnnData
DE_ANALYZED_ANNDATA_FILE = config.PROCESSED_DATA_DIR / "04_Norman_2019_DE_analyzed.h5ad"
print("\nDE-analyzed AnnData file will be saved to:\n", DE_ANALYZED_ANNDATA_FILE)

# DE Analysis results CSV file
DE_RESULTS_CSV_DIR = config.RESULTS_DIR / "04_de_analysis"
DE_RESULTS_CSV_DIR.mkdir(parents=True, exist_ok=True)
print("\nDE Analysis results CSV files will be saved to:\n", DE_RESULTS_CSV_DIR)

# Figures save directory
FIGURES_DIR = config.FIGURES_DIR / "04_de_analysis"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
print("\nFigures will be saved to:\n", FIGURES_DIR)

### 1.3. Define Global Parameters

In [ ]:
SEED = config.SEED

np.random.seed(SEED)
sc.settings.seed = SEED

print(f"\nRandom seed set to: {SEED}")

---

## 2. Load And Validate Data

### 2.1. Load Clustered AnnData

In [ ]:
# Read the clustered AnnData file
print("\nReading clustered AnnData file...")
adata = sc.read_h5ad(CLUSTERED_ANNDATA_FILE)
print(adata)

### 2.2. Inspect Data

In [ ]:
print("Cell IDs:\n", adata.obs_names)
print("\nGene IDs:\n", adata.var_names)

print("\nGene symbols:\n", adata.var.gene_symbols)

print("\nSample data matrix (first 5 rows and columns):\n", adata.X[:5, :5].toarray())

### 2.3. Validate Data

#### 2.3.1. Verify Normalization and HVG Selection

In [ ]:
# Verify normalization
x_max = adata.X.data.max()
x_min = adata.X.data.min()
assert x_min >= 0, "Negative values found — normalization error"
assert (
    x_max <= np.log1p(1e4) + 0.01
), f"Max value {x_max:.3f} exceeds log1p(10000) — log1p may not have been applied"
print(f"Normalization and log1p verified (value range: {x_min:.3f} - {x_max:.3f})")

# Verify raw counts layer preserved
assert "counts" in adata.layers, "Raw counts layer missing"
assert np.all(
    adata.layers["counts"].data == np.floor(adata.layers["counts"].data)
), "Raw counts layer contains non-integers"
print("Raw counts layer preserved")

# Verify HVGs selected
assert adata.var["highly_variable"].sum() == 3000, "HVG count mismatch"
print("HVGs selected: 3000")

print("\nAll assertions passed.")

#### 2.3.2. Index Genes By Symbol
Index genes by their symbols instead of Ensembl IDs.

In [ ]:
# Check if genes are already indexed by their symbols
are_genes_ens_ids = adata.var_names.str.startswith("ENSG").all()

# Map to gene symbols if currently indexed by Ensembl IDs
if are_genes_ens_ids:
    print("\nGenes are indexed by Ensembl IDs. Mapping to gene symbols....")

    # Check if gene symbols are unique
    if adata.var.gene_symbols.is_unique:
        print("Gene symbols are unique. Setting gene symbols as index....")
        adata.var_names = adata.var['gene_symbols'].values
        print("\nGene symbols set as index successfully.")
    else:
        print("\nGene symbols are NOT unique.")

        # How many duplicates?
        duplicates = adata.var['gene_symbols'][adata.var['gene_symbols'].duplicated(keep=False)]
        print(f"Number of duplicate gene symbol entries: {len(duplicates)}")
        print(duplicates.value_counts().head())

        # Make gene symbols unique by appending a suffix to duplicates
        print("\nMaking gene symbols unique and setting them as index....")
        adata.var_names = adata.var['gene_symbols'].values
        adata.var_names_make_unique()
        print("Gene symbols made unique and set as index successfully.")

# If genes are already indexed by gene symbols, no mapping is needed
else:
    print("\nGenes are already indexed by gene symbols. No mapping needed.")

print("\nGene IDs:\n", adata.var_names)

#### 2.3.3. Verify Uniqueness Of Cell IDs

In [ ]:
are_cell_ids_unique = adata.obs_names.is_unique

if are_cell_ids_unique:
    print("\nCell IDs are unique.")
else:
    print("\nFAILED: Cell IDs are NOT unique.")

#### 2.3.4. Verify Clustering

In [ ]:
# Verify PCA
assert "X_pca" in adata.obsm, "PCA not found"
print(f"PCA present: {adata.obsm['X_pca'].shape[1]} components")

# Verify neighbors graph
assert "neighbors" in adata.uns, "Neighbors graph not found"
assert adata.uns["neighbors"]["params"]["n_pcs"] == 10, "n_pcs mismatch"
print(f"Neighbors graph present: n_pcs=10")

# Verify UMAP
assert "X_umap" in adata.obsm, "UMAP not found"
print(f"UMAP present: {adata.obsm['X_umap'].shape[1]} dimensions")

# Verify leiden
assert "leiden" in adata.obs.columns, "Leiden clusters not found"
assert (
    adata.obs["leiden"].nunique() == 8
), f"Expected 8 clusters, got {adata.obs['leiden'].nunique()}"
print(f"Leiden clusters present: {adata.obs['leiden'].nunique()} clusters")

# Verify cell cycle
assert "phase" in adata.obs.columns, "Cell cycle phase missing"
assert set(adata.obs["phase"].unique()) == {"G1", "S", "G2M"}, "Unexpected phase labels"
print(f"Cell cycle phases present: {adata.obs['phase'].value_counts().to_dict()}")

# Verify perturbation columns
assert "perturbation_clean" in adata.obs.columns, "perturbation_clean missing"
assert "CEBPE" in adata.obs["perturbation_clean"].values, "CEBPE not found"
assert "Control" in adata.obs["perturbation_clean"].values, "Control not found"
print(
    f"Perturbation labels present: CEBPE ({(adata.obs['perturbation_clean']=='CEBPE').sum():,} cells), Control ({(adata.obs['perturbation_clean']=='Control').sum():,} cells)"
)

print("\nAll assertions passed.")

---

## 3. Perform DE Analysis

### 3.1. Subset To CEBPE And Control Cells

In [ ]:
print("Subsetting to CEBPE and Control cells...")
adata_sub = adata[adata.obs["perturbation_clean"].isin(["CEBPE", "Control"])].copy()
print(
    f"Subset: {adata_sub.n_obs:,} cells (CEBPE: {(adata_sub.obs['perturbation_clean']=='CEBPE').sum():,}, Control: {(adata_sub.obs['perturbation_clean']=='Control').sum():,})"
)

### 3.2. Prefilter Genes By Expression Percentage
Retain only genes expressed in at least 10% of cells in either the CEBPE or Control group, removing sparsely expressed genes that would otherwise produce unreliable DE statistics.

In [ ]:
print("Prefiltering genes by expression percentage...")
cebpe_mask_np = (adata_sub.obs["perturbation_clean"] == "CEBPE").values
control_mask_np = (adata_sub.obs["perturbation_clean"] == "Control").values

cebpe_counts = adata_sub.layers["counts"][cebpe_mask_np]
control_counts = adata_sub.layers["counts"][control_mask_np]

pct_cebpe_all = np.array((cebpe_counts > 0).mean(axis=0)).flatten()
pct_control_all = np.array((control_counts > 0).mean(axis=0)).flatten()

gene_mask = (pct_cebpe_all >= 0.10) | (pct_control_all >= 0.10)
adata_sub = adata_sub[:, gene_mask].copy()

print(
    f"Genes retained: {adata_sub.n_vars:,} / {gene_mask.shape[0]:,} ({gene_mask.mean()*100:.1f}%)"
)

### 3.3. Run Wlicoxon Rank-Sum DE Test

In [ ]:
print("\nRunning Wilcoxon rank-sum DE test (CEBPE vs Control)...")
sc.tl.rank_genes_groups(
    adata_sub,
    groupby="perturbation_clean",
    groups=["CEBPE"],
    reference="Control",
    method="wilcoxon",
    corr_method="benjamini-hochberg",
    tie_correct=True,
    key_added="de_cebpe_vs_control",
)
print("DE test complete.")

### 3.4. Extract DE Results Into DataFrame

In [ ]:
# Extract DE results into dataframe
print("Extracting DE results...")
de_results = sc.get.rank_genes_groups_df(
    adata_sub,
    group="CEBPE",
    key="de_cebpe_vs_control",
    pval_cutoff=None,
    log2fc_min=None,
)

print(f"Total genes tested: {len(de_results):,}")
print(f"\nTop 10 upregulated genes:")
print(
    de_results.nlargest(10, "logfoldchanges")[
        ["names", "logfoldchanges", "pvals", "pvals_adj"]
    ]
)
print(f"\nTop 10 downregulated genes:")
print(
    de_results.nsmallest(10, "logfoldchanges")[
        ["names", "logfoldchanges", "pvals", "pvals_adj"]
    ]
)

### 3.5. Filter By Statistical Significance And Fold Change

In [ ]:
print("\nFiltering DE results by adjusted p-value < 0.05 and absolute log1p (ln) fold change > 0.5...")
de_filtered = de_results[
    (de_results["pvals_adj"] < 0.05) & (de_results["logfoldchanges"].abs() > 0.5)
].copy()

print(f"Total DE genes after filtering: {len(de_filtered):,}")
print(f"Upregulated: {(de_filtered['logfoldchanges'] > 0).sum():,}")
print(f"Downregulated: {(de_filtered['logfoldchanges'] < 0).sum():,}")

de_filtered = de_filtered.copy()
de_filtered["log2fc"] = de_filtered["logfoldchanges"] / np.log(2)
de_filtered["actual_fold_change"] = np.exp(de_filtered["logfoldchanges"])

print("\nTop 10 upregulated genes:")
print(
    de_filtered.nlargest(10, "actual_fold_change")[
        ["names", "actual_fold_change", "log2fc", "logfoldchanges", "pvals_adj"]
    ].to_string(index=False)
)

print("\nTop 10 downregulated genes:")
print(
    de_filtered.nsmallest(10, "actual_fold_change")[
        ["names", "actual_fold_change", "log2fc", "logfoldchanges", "pvals_adj"]
    ].to_string(index=False)
)

The column **`actual_fold_change`** expresses how much more or less a gene is expressed in CEBPE-perturbed cells relative to control cells in linear scale:

- **`actual_fold_change` = 2626** → gene is **2626x higher** in CEBPE cells
- **`actual_fold_change` = 1413** → gene is **1413x higher** in CEBPE cells
- **`actual_fold_change` = 0.086** → gene is **1/0.086 = 11.63x lower** in CEBPE cells
- **`actual_fold_change` = 0.158** → gene is **1/0.158 = 6.33x lower** in CEBPE cells

**`log2fc`** is the same quantity in log2 scale, where a difference of 1.0 always represents a doubling regardless of baseline expression level — useful for visualization. **`logfoldchanges`** is the raw output from scanpy's rank-sum test in natural log (ln) scale, a direct consequence of the log1p transformation applied during preprocessing. All three columns represent the same underlying quantity in different scales.

### 3.6. Visualize DE Results (Volcano Plot)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 7))

# All genes — grey background
ax.scatter(
    de_results["logfoldchanges"] / np.log(2),
    -np.log10(de_results["pvals_adj"].replace(0, 1e-320)),
    c="lightgrey",
    s=1,
    alpha=0.5,
    label="Not significant",
)

# Significant upregulated — red
up_mask = de_filtered["log2fc"] > 0
ax.scatter(
    de_filtered.loc[up_mask, "log2fc"],
    -np.log10(de_filtered.loc[up_mask, "pvals_adj"].replace(0, 1e-320)),
    c="red",
    s=5,
    alpha=0.7,
    label=f"Upregulated ({up_mask.sum():,})",
)

# Significant downregulated — blue
down_mask = de_filtered["log2fc"] < 0
ax.scatter(
    de_filtered.loc[down_mask, "log2fc"],
    -np.log10(de_filtered.loc[down_mask, "pvals_adj"].replace(0, 1e-320)),
    c="blue",
    s=5,
    alpha=0.7,
    label=f"Downregulated ({down_mask.sum():,})",
)

# Label top 10 upregulated and top 5 downregulated
top_genes = pd.concat(
    [de_filtered.nlargest(10, "log2fc"), de_filtered.nsmallest(5, "log2fc")]
)
for _, row in top_genes.iterrows():
    ax.annotate(
        row["names"],
        xy=(row["log2fc"], -np.log10(max(row["pvals_adj"], 1e-320))),
        fontsize=7,
        ha="left",
    )

ax.axvline(x=0.5 / np.log(2), color="black", linestyle="--", linewidth=0.8)
ax.axvline(x=-0.5 / np.log(2), color="black", linestyle="--", linewidth=0.8)
ax.axhline(y=-np.log10(0.05), color="black", linestyle="--", linewidth=0.8)

ax.set_xlabel("log2 Fold Change (CEBPE vs Control)")
ax.set_ylabel("-log10(Adjusted P-value)")
ax.set_title("Volcano Plot: CEBPE vs Control")
ax.legend(markerscale=3)

plt.tight_layout()
plt.savefig(str(FIGURES_DIR / "01_de_volcano_plot.png"), bbox_inches="tight")
plt.show()

### 3.7. Quantify Cell Cycle Contribution To DE Results
While perturbation is the primary driver of DE results by experimental design, cell cycle phase is a known confounder in dividing cell lines. The overlap between DE genes and known cell cycle gene lists is quantified to assess the extent of potential cell cycle confounding.

#### 3.7.1. Compute Overlap Between DE and Cell Cycle Genes

In [ ]:
# Tirosh et al. 2016 cell cycle gene lists
cell_cycle_gene_list = [
    # S phase genes
    'MCM5', 'PCNA', 'TYMS', 'FEN1', 'MCM2', 'MCM4', 'RRM1', 'UNG',
    'GINS2', 'MCM6', 'CDCA7', 'DTL', 'PRIM1', 'UHRF1', 'MLF1IP',
    'HELLS', 'RFC2', 'RPA2', 'NASP', 'RAD51AP1', 'GMNN', 'WDR76',
    'SLBP', 'CCNE2', 'UBR7', 'POLD3', 'MSH2', 'ATAD2', 'RAD51',
    'RRM2', 'CDC45', 'CDC6', 'EXO1', 'TIPIN', 'DSCC1', 'BLM',
    'CASP8AP2', 'USP1', 'CLSPN', 'POLA1', 'CHAF1B', 'BRIP1', 'E2F8',

    # G2/M phase genes
    'HMGB2', 'CDK1', 'NUSAP1', 'UBE2C', 'BIRC5', 'TPX2', 'TOP2A',
    'NDC80', 'CKS2', 'NUF2', 'CKS1B', 'MKI67', 'TMPO', 'CENPF',
    'TACC3', 'FAM64A', 'SMC4', 'CCNB2', 'CKAP2L', 'CKAP2', 'AURKB',
    'BUB1', 'KIF11', 'ANP32E', 'TUBB4B', 'GTSE1', 'KIF20B', 'HJURP',
    'CDCA3', 'HN1', 'CDC20', 'TTK', 'CDC25C', 'KIF2C', 'RANGAP1',
    'NCAPD2', 'DLGAP5', 'CDCA2', 'CDCA8', 'ECT2', 'KIF23', 'HMMR',
    'AURKA', 'PSRC1', 'ANLN', 'LBR', 'CKAP5', 'CENPE', 'CTCF',
    'NEK2', 'G2E3', 'GAS2L3', 'CBX5', 'CENPA'
]

cell_cycle_genes = set(cell_cycle_gene_list)
de_genes = set(de_filtered["names"])  # All DE genes after filtering; might overlap with cell cycle genes

print("Computing overlap between DE genes and known cell cycle genes...")
overlap = cell_cycle_genes & de_genes
overlap_pct = len(overlap) / len(de_genes) * 100

print(f"Total DE genes: {len(de_genes):,}")
print(f"Known cell cycle genes: {len(cell_cycle_genes):,}")
print(f"Overlap: {len(overlap):,} genes ({overlap_pct:.1f}% of DE genes)")
print(f"\nOverlapping genes: {sorted(overlap)}")

#### 3.7.2. Analyse S Phase vs G2/M Phase Genes

In [ ]:
# Check direction of overlapping genes
overlap_df = de_filtered[de_filtered["names"].isin(overlap)][["names", "actual_fold_change", "log2fc", "pvals_adj"]].sort_values("log2fc")
print("Direction of cell cycle gene overlap:")
print(overlap_df.sort_values("actual_fold_change", ascending=False).to_string(index=False))

# S phase vs G2M breakdown
s_gene_set = set(['MCM5', 'PCNA', 'TYMS', 'FEN1', 'MCM2', 'MCM4', 'RRM1', 'UNG',
    'GINS2', 'MCM6', 'CDCA7', 'DTL', 'PRIM1', 'UHRF1', 'MLF1IP',
    'HELLS', 'RFC2', 'RPA2', 'NASP', 'RAD51AP1', 'GMNN', 'WDR76',
    'SLBP', 'CCNE2', 'UBR7', 'POLD3', 'MSH2', 'ATAD2', 'RAD51',
    'RRM2', 'CDC45', 'CDC6', 'EXO1', 'TIPIN', 'DSCC1', 'BLM',
    'CASP8AP2', 'USP1', 'CLSPN', 'POLA1', 'CHAF1B', 'BRIP1', 'E2F8'])
g2m_gene_set = cell_cycle_genes - s_gene_set

s_overlap = overlap & s_gene_set
g2m_overlap = overlap & g2m_gene_set

print(f"\nS phase genes in overlap: {len(s_overlap)} — {sorted(s_overlap)}")
print(f"G2M genes in overlap: {len(g2m_overlap)} — {sorted(g2m_overlap)}")

Of the 1,228 DE genes, only **11 (0.9%)** overlap with known cell cycle genes from the Tirosh et al. 2016 gene lists. Critically, all 11 are **downregulated** in CEBPE-perturbed cells (actual fold change range: 0.53–0.60, i.e. expressed at roughly half the level of control cells), with G2M genes comprising the majority of the overlap (7 of 11).

This pattern is biologically coherent rather than artifactual — CEBPE overactivation drives granulocytic differentiation, which involves cell cycle exit. Downregulation of G2M genes is an expected consequence of cells transitioning toward G1 arrest and differentiation, not evidence of cell cycle confounding.

The remaining **99.1% of DE genes** are therefore interpreted as predominantly reflecting direct or indirect transcriptional consequences of **CEBPE overactivation**, independent of cell cycle phase differences between groups.

## 4. Perform Final Sanity Checks

In [ ]:
print("=== POST-DE SANITY CHECK ===\n")

# Verify subset was correct
assert set(adata_sub.obs["perturbation_clean"].unique()) == {
    "CEBPE",
    "Control",
}, "Unexpected groups in subset"
assert adata_sub.n_obs == 11693, f"Expected 11,693 cells, got {adata_sub.n_obs}"
print(f"Subset verified: {adata_sub.n_obs:,} cells (CEBPE + Control only)")

# Verify gene prefiltering
assert (
    adata_sub.n_vars == 8077
), f"Expected 8,077 genes after prefiltering, got {adata_sub.n_vars}"
print(f"Gene prefiltering verified: {adata_sub.n_vars:,} genes retained")

# Verify DE test was run
assert "de_cebpe_vs_control" in adata_sub.uns, "DE results not found in adata_sub.uns"
print(f"DE test results present in adata_sub.uns")

# Verify DE results dataframe
assert "log2fc" in de_filtered.columns, "log2fc column missing"
assert "actual_fold_change" in de_filtered.columns, "actual_fold_change column missing"
assert "pvals_adj" in de_filtered.columns, "pvals_adj column missing"
print(f"DE results columns verified")

# Verify filtering thresholds were applied
assert (
    de_filtered["pvals_adj"] < 0.05
).all(), "Some genes exceed adjusted p-value threshold"
assert (
    de_filtered["logfoldchanges"].abs() > 0.5
).all(), "Some genes below fold change threshold"
print(f"Filtering thresholds verified")

# Verify DE gene counts
assert len(de_filtered) == 1228, f"Expected 1,228 DE genes, got {len(de_filtered)}"
assert (de_filtered["logfoldchanges"] > 0).sum() == 604, "Upregulated count mismatch"
assert (de_filtered["logfoldchanges"] < 0).sum() == 624, "Downregulated count mismatch"
print(f"DE genes verified: {len(de_filtered):,} total (604 up, 624 down)")

# Verify cell cycle overlap
cc_overlap_count = de_filtered["names"].isin(cell_cycle_genes).sum()
assert (
    cc_overlap_count == 11
), f"Expected 11 cell cycle gene overlaps, got {cc_overlap_count}"
print(f"Cell cycle overlap verified: {cc_overlap_count} genes (0.9%)")

---

## 5. Save Results

### 5.1. Save DE-Analyzed AnnData

In [ ]:
adata.write_h5ad(DE_ANALYZED_ANNDATA_FILE)
print(f"Saved: {DE_ANALYZED_ANNDATA_FILE}")

### 5.2. Save DE Analysis Results

#### 5.2.1. Split Results And Sort By Combined Rank
Split the gene list into upregulated and downregulated genes, and sort both by `combined_rank`, a product of the absolute values in the `scores` and `logfoldchanges` columns.

In [ ]:
# Add combined rank metric
de_filtered["combined_rank"] = (
    de_filtered["scores"].abs() * de_filtered["logfoldchanges"].abs()
)

# Split into upregulated and downregulated
upregulated = de_filtered[de_filtered["logfoldchanges"] > 0].sort_values(
    "combined_rank", ascending=False
)
downregulated = de_filtered[de_filtered["logfoldchanges"] < 0].sort_values(
    "combined_rank", ascending=False
)

print(f"Upregulated: {len(upregulated):,}")
print(f"Downregulated: {len(downregulated):,}")

print("\nTop 10 upregulated genes (by combined rank):")
print(
    upregulated.head(10)[
        ["names", "scores", "logfoldchanges", "log2fc", "combined_rank", "pvals_adj"]
    ].to_string(index=False)
)

print("\nTop 10 downregulated genes (by combined rank):")
print(
    downregulated.head(10)[
        ["names", "scores", "logfoldchanges", "log2fc", "combined_rank", "pvals_adj"]
    ].to_string(index=False)
)

#### 5.2.2. Save Upregulated Gene List

In [ ]:
upregulated.to_csv(
    DE_RESULTS_CSV_DIR / "01_cebpe_vs_control_de_results_up.csv", index=False
)
print(f"Saved: {DE_RESULTS_CSV_DIR / '01_cebpe_vs_control_de_results_up.csv'}")

#### 5.2.3. Save Downregulated Gene List

In [ ]:
downregulated.to_csv(
    DE_RESULTS_CSV_DIR / "02_cebpe_vs_control_de_results_down.csv", index=False
)
print(f"Saved: {DE_RESULTS_CSV_DIR / '02_cebpe_vs_control_de_results_down.csv'}")

---

## 6. Summary

### 6.1. Data Validation
- Gene symbols confirmed as index
- Cell barcodes confirmed unique
- Normalization, log1p transformation, and raw counts layer verified
- Clustering outputs (perturbation_clean, phase, leiden) confirmed present

### 6.2. Steps Executed
1. **Subset** — restricted analysis to CEBPE (1,101 cells) and Control (10,592 cells) populations
2. **Gene prefiltering** — retained 8,077 / 19,996 genes expressed in ≥10% of cells in either group, removing sparsely expressed genes that produce unreliable DE statistics
3. **Wilcoxon rank-sum DE test** — non-parametric two-group comparison of CEBPE vs Control using Benjamini-Hochberg multiple testing correction
4. **Results extraction** — DE results extracted with `logfoldchanges` (ln/natural log scale), `log2fc` (log2 scale), and `actual_fold_change` columns computed
5. **Filtering** — retained genes with adjusted `p-value` < 0.05 and absolute ln (natural log) fold change > 0.5
6. **Volcano plot** — visualised DE results with `log2fc` on x-axis and -log10(adjusted `p-value`) on y-axis
7. **Cell cycle contribution** — quantified overlap between DE genes and Tirosh et al. 2016 cell cycle gene lists

### 6.3. Key Findings
- **1,228 DE genes** identified (604 upregulated, 624 downregulated) in CEBPE vs Control
- **CEBPE itself** is the second most upregulated gene (actual fold change ~1,413x) — confirms CRISPRa activation worked correctly
- **Top upregulated genes** include TRGC1, PLEK, FYB, HCLS1, PTPRC, CD84 — all hematopoietic/myeloid markers consistent with granulocytic differentiation
- **Top downregulated genes** include HEMGN, GYPA, GYPB, RHAG — erythroid markers, consistent with CEBPE suppressing alternative blood cell fates
- **Cell cycle confounding is minimal** — only 11 DE genes (0.9%) overlap with known cell cycle genes, all downregulated and consistent with differentiation-driven cell cycle slowing rather than artifactual confounding

### 6.4. Parameters
- Subset: CEBPE vs Control only
- Gene prefilter: ≥10% of cells expressing in either group
- DE method: Wilcoxon rank-sum, Benjamini-Hochberg correction, `tie_correct=True`
- Filter thresholds: adjusted `p-value` < 0.05, absolute ln fold change > 0.5

### 6.5. AnnData Modifications
**`adata_sub.uns` (unstructured results):** - `de_cebpe_vs_control` — full Wilcoxon rank-sum DE results for all 8,077 tested genes

### 6.6. DE Results
- `01_cebpe_vs_control_de_results_up.csv`
- `02_cebpe_vs_control_de_results_down.csv`